# Feedstock Quality Assessment & Steel Defect Detection (Industry 4.0)
## Vision Transformer (ViT) End-to-End Pipeline

This notebook provides a complete local pipeline for:
1. Dataset loading and inspection (Severstal Steel Defect Detection benchmark).
2. Run-Length Encoding (RLE) mask decoding and spatial contour analysis.
3. Multi-Head Self-Attention Vision Transformer (ViT) inference.
4. Attention Rollout Heatmap generation and defect localization.
5. Industry 4.0 automated quality grading (Grade A / B / C) and recommendation directives.

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch

# Ensure models and backend are in path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from models.vit_model import FeedstockVisionTransformer
from models.defect_classifier import FeedstockDefectDetector
from models.rle_codec import rle_to_mask, mask_to_rle, extract_defect_regions, compute_mask_metrics

print(f"PyTorch Version: {torch.__version__}")
print(f"Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

### 1. Load Dataset Metadata & Inspect Benchmark Samples

In [ ]:
csv_path = PROJECT_ROOT / "data" / "severstal-steel-defect-detection" / "train.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    print(f"Total Defect Records: {len(df):,}")
    print("\nDefect Class Breakdown:")
    print(df['ClassId'].value_counts())
    display_df = df.head()
else:
    print("Dataset train.csv not found, using samples metadata.")

meta_path = PROJECT_ROOT / "data" / "samples_metadata.json"
with open(meta_path, 'r', encoding='utf-8') as f:
    samples = json.load(f)

print(f"\nLoaded {len(samples)} Curated Benchmark Samples:")
for s in samples[:5]:
    print(f"- {s['filename']} -> {s['class_name']} ({s['quality_grade']})")

### 2. Run-Length Encoding (RLE) Decoding & Visualization

In [ ]:
# Select a defective sample
sample = next(s for s in samples if s['class_id'] == 1)
img_path = PROJECT_ROOT / "data" / "samples" / sample['filename']
img = cv2.imread(str(img_path))
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

mask = rle_to_mask(sample['encoded_pixels'], shape=img.shape[:2])
metrics = compute_mask_metrics(mask, shape=img.shape[:2])

plt.figure(figsize=(15, 4))
plt.subplot(2, 1, 1)
plt.imshow(img_rgb)
plt.title(f"Raw Feedstock Image: {sample['filename']}")
plt.axis('off')

plt.subplot(2, 1, 2)
plt.imshow(mask, cmap='hot')
plt.title(f"Decoded Defect Mask (Defect Area: {metrics['defect_area_pct']}% | Clusters: {metrics['regions_count']})")
plt.axis('off')
plt.tight_layout()
plt.show()

### 3. Initialize Vision Transformer & Run Defect Analysis

In [ ]:
detector = FeedstockDefectDetector()
result = detector.analyze_image(
    str(img_path),
    ground_truth_rle=sample['encoded_pixels'],
    ground_truth_class=sample['class_id']
)

print("=" * 50)
print(f"VERDICT: {result['verdict']}")
print(f"QUALITY GRADE: {result['quality_grade']}")
print(f"DEFECT CLASS: {result['class_name']}")
print(f"SEVERITY: {result['severity']}")
print(f"QUALITY SCORE: {result['quality_score']} / 100")
print(f"AFFECTED AREA: {result['defect_area_pct']}%")
print(f"DIRECTIVE: {result['class_metadata']['recommended_action']}")
print("=" * 50)

### 4. Bounding Box & Defect Contours Summary

In [ ]:
print(f"Total Defect Regions Detected: {len(result['regions'])}")
for r in result['regions'][:5]:
    b = r['bbox']
    print(f"- Region #{r['region_id']}: BBox [X={b['x']}, Y={b['y']}, W={b['width']}, H={b['height']}] | Area={r['area_pixels']} px")